In [ ]:
import torch
from training_functions import train, evaluate_mae_mse
from TransCrowd import base_patch16_384_token, base_patch16_384_gap
import torch.nn as nn
import torch.optim as optim
from dataloader_mcnn import CrowdDataset
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, ConcatDataset, Subset
import glob
import os
import numpy as np
import random
import time
import pandas as pd
from torchvision import transforms
SEED = 42

In [ ]:
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)

### Model training

In [ ]:
main_folders = [r"data\ShanghaiTech_part_A", 
                r"data\ShanghaiTech_part_B"]

train_datasets = []
val_datasets = []


transform = transforms.Compose([
                                transforms.ToTensor(),

                                transforms.Normalize(mean=[0.485, 0.456, 0.406],
                                                     std=[0.229, 0.224, 0.225]),
                            ])

for folder in main_folders:
    train_img_root = os.path.join(folder, "train", "images")
    train_map_root = os.path.join(folder, "train", "maps")
    print(folder)
    if os.path.exists(train_img_root) and os.path.exists(train_map_root):
        print('ok')
        train_datasets.append(CrowdDataset(train_img_root, train_map_root, gt_downsample=1, resize=False,
                                           transform=transform, patch=True, train=True))
    
    val_img_root = os.path.join(folder, "test", "images")
    val_map_root = os.path.join(folder, "test", "maps")
    
    if os.path.exists(val_img_root) and os.path.exists(val_map_root):
        val_datasets.append(CrowdDataset(val_img_root, val_map_root, gt_downsample=1, resize=False,
                                         transform=transform, patch=True))

train_dataset = ConcatDataset(train_datasets)
val_dataset = ConcatDataset(val_datasets)

train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True, pin_memory=True, num_workers = 2)
val_loader = DataLoader(val_dataset, batch_size=4, shuffle=False,pin_memory=True, num_workers = 2)

In [ ]:
set_seed(SEED)
model_dir = r"output\models\transcrowd"
history_dir = r"output\history\transcrowd"

os.makedirs(model_dir, exist_ok=True)
os.makedirs(history_dir, exist_ok=True)

model_path = model_dir + "/transcrowd.pth"
history_path = history_dir + "/transcrowd.pkl"

device = "cuda" if torch.cuda.is_available() else "cpu"
model = base_patch16_384_gap(pretrained=True, path = r'output\models\transcrowd\deit_base_patch16_384-8de9b5d1.pth')
model = model.to(device)
epochs = 100
tolerance = 20
print(device)
criterion = nn.SmoothL1Loss(reduction='mean').to(device)
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-5,
    weight_decay=1e-4
)


start_time = time.time()
history = train(epochs, model, train_loader, val_loader, optimizer, criterion, device,model_path, tolerance = tolerance, is_dmap=False)
end_time = time.time()
print(f"total time: {end_time - start_time}\n")


### Fine tuning

In [ ]:
main_folders = [
    r"data\ShanghaiTech_part_A\train", 
    r"data\ShanghaiTech_part_B\train",
    r"data\NWPU-Crowd\train",
    r"data\UCF-QNRF_ECCV18\train",
    r"data\jhu_crowd_v2.0\train"
]

train_datasets = []
val_datasets = []

transform = transforms.Compose([
                                transforms.ToTensor(),

                                transforms.Normalize(mean=[0.485, 0.456, 0.406],
                                                     std=[0.229, 0.224, 0.225]),
                            ])

for folder in main_folders:

    train_img_root = os.path.join(folder, "images")
    train_map_root = os.path.join(folder, "maps")

    train_datasets.append(
        CrowdDataset(
            train_img_root,
            train_map_root,
            gt_downsample=1,
            resize=False,
            transform=transform,
            patch=True,
            train=False
        )
    )

    dataset_root = os.path.dirname(folder)
    if 'NWPU' in dataset_root:
        val_img_root = os.path.join(dataset_root, "val", "images")
        val_map_root = os.path.join(dataset_root, "val", "maps")
    else:
        val_img_root = os.path.join(dataset_root, "test", "images")
        val_map_root = os.path.join(dataset_root, "test", "maps")

    val_datasets.append(
        CrowdDataset(
            val_img_root,
            val_map_root,
            gt_downsample=1,
            resize=False,
            transform=transform,
            patch=True
        )
    )

In [ ]:
set_seed(SEED)

model_dir = r"output\models\transcrowd"
history_dir = r"output\history\transcrowd"

os.makedirs(model_dir, exist_ok=True)
os.makedirs(history_dir, exist_ok=True)

model_path = model_dir + r"\transcrowd_finetuned.pth"
history_path = history_dir + r"\transcrowd.pkl"

device = "cuda" if torch.cuda.is_available() else "cpu"

model = base_patch16_384_gap(
    pretrained=True,
    path=r'output\models\transcrowd\deit_base_patch16_384-8de9b5d1.pth'
)

model.load_state_dict(
    torch.load(
        r'output\models\transcrowd\transcrowd.pth'
    )
)

model = model.to(device)

criterion = nn.SmoothL1Loss(reduction="mean").to(device)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-5,
    weight_decay=1e-4
)

print(device)

In [ ]:
num_iterations = 300
epochs_per_iteration = 3
samples_per_iteration = 50
batch_size = 4

dataset_counts = [0] * len(train_datasets)

start_time = time.time()

for iteration in range(num_iterations):

    dataset_idx = random.randrange(len(train_datasets))
    dataset = train_datasets[dataset_idx]

    dataset_counts[dataset_idx] += 1
    n_samples = min(samples_per_iteration, len(dataset))

    indices = random.sample(
        range(len(dataset)),
        n_samples
    )

    sampled_dataset = Subset(
        dataset,
        indices
    )

    indices_val = random.sample(
        range(len(val_datasets[dataset_idx])),
        n_samples//5
    )

    sampled_dataset_val = Subset(
        val_datasets[dataset_idx],
        indices_val
    )

    train_loader = DataLoader(
        sampled_dataset,
        batch_size=batch_size,
        shuffle=True,
        pin_memory=True,
        num_workers=2
    )
    val_loader = DataLoader(sampled_dataset_val, batch_size=4, shuffle=False,pin_memory=True, num_workers = 2)

    print(
        f"\nIteration {iteration + 1}/{num_iterations} "
        f"| Dataset {dataset_idx} "
    )

    history = train(
        epochs_per_iteration,
        model,
        train_loader,
        val_loader,             
        optimizer,
        criterion,
        device,
        model_path,
        tolerance=10,
        is_dmap=False
    )

    print(
        "Dataset counts:",
        dataset_counts
    )


end_time = time.time()

print("Final dataset counts:")

for i, count in enumerate(dataset_counts):
    print(f"Dataset {i}: {count} iterations")